# Step 2: Feature Extraction & Dataset Construction

**Paper:** Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks  
**Author:** Kenny Ching — School of Business, University of Auckland  

---

### Description
This notebook constructs the master econometric dataset (`tf_events_master_unified_v2.csv`)
from two sources:

**Part A — AI Observations (from .dem replay files):**  
Parses the combatlog text files generated in Step 1 for the three OpenAI Five matches
(Game 1, Game 2, Coop). Extracts teamfights, gold timelines, and tactical shock events
for the AI agent.

**Part B — Human Baseline (from OpenDota API):**  
Downloads match telemetry for TI8 and TI9 Main Event matches via the OpenDota public API.
Extracts the same tactical shock variables for the human cohort.

**Part C — Merge:**  
Combines both sources into a single unified CSV with consistent schema.

### Input
- `og_game1_combatlog.txt`, `og_game2_combatlog.txt`, `coop_combatlog.txt` (from Step 1)

### Output
- `tf_events_master_unified_v2.csv` — 9,228 rows (148 AI + 9,080 human)

### Schema
| Column | Description |
|---|---|
| `match_id` | Match identifier |
| `shock_type` | `negative` or `positive` |
| `is_ai` | 1 = OpenAI Five, 0 = human |
| `did_win` | 1 = focal team won the match |
| `fight_start` | Game clock time (seconds) at fight start |
| `current_gold_lead` | Focal team net worth advantage at fight start |
| `gadv_d30/60/120/180` | Change in gold advantage at t+30/60/120/180s |

In [ ]:
# ============================================================
# CELL 1 — Imports and upload combatlog files
# ============================================================
import os, re, json, time
import requests
import pandas as pd
import numpy as np
from collections import defaultdict
from google.colab import files

HORIZONS = [30, 60, 120, 180]
OUT_DIR  = '/content/data'
os.makedirs(OUT_DIR, exist_ok=True)

print('Upload the 3 combatlog .txt files from Step 1:')
uploaded = files.upload()

COMBATLOG_PATHS = {}
for fname in uploaded:
    dest = f'/content/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    fl = fname.lower()
    if 'game1' in fl or 'game_1' in fl:
        COMBATLOG_PATHS['game1'] = dest
    elif 'game2' in fl or 'game_2' in fl:
        COMBATLOG_PATHS['game2'] = dest
    elif 'coop' in fl:
        COMBATLOG_PATHS['coop'] = dest

print('\nDetected:')
for k,v in COMBATLOG_PATHS.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# CELL 2 — Combatlog parser (Part A: AI observations)
# ============================================================

# --- Regex patterns ---
TS       = r'\[(\d{2}):(\d{2}):(\d{2})\.(\d{3})\]'
STATE_RE = re.compile(TS + r' game state is now (\d+)')
KILL_RE  = re.compile(TS + r' (npc_dota_hero_\w+) is killed by (npc_dota_hero_\w+)')
RECV_RE  = re.compile(TS + r' (npc_dota_hero_\w+) receives (\d+) gold')
LOSS_RE  = re.compile(TS + r' (npc_dota_hero_\w+) looses (\d+) gold')  # note: game bug spelling

def ts2s(h, m, s, ms):
    return int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000.0

def short(hero):
    return hero.replace('npc_dota_hero_', '')

# --- Game configurations ---
# Team assignments verified from kill patterns + ancient destruction in combat log.
# 'goodguys' (Radiant) creeps destroying 'badguys_fort' = Radiant victory.
GAME_CFG = {
    'game1': {
        'match_id':     4132013569,
        'label':        'OG vs OpenAI Five - Game 1',
        'radiant_win':  True,
        'radiant':      {'earthshaker','viper','witch_doctor','nevermore','sven'},
        'dire':         {'gyrocopter','crystal_maiden','riki','death_prophet','sniper'},
        'ai_side':      'dire',    # OpenAI Five played Dire; they lost
    },
    'game2': {
        'match_id':     4132030946,
        'label':        'OG vs OpenAI Five - Game 2',
        'radiant_win':  True,
        'radiant':      {'viper','gyrocopter','sven','crystal_maiden','witch_doctor'},
        'dire':         {'death_prophet','slark','lion','earthshaker','sniper'},
        'ai_side':      'radiant', # OpenAI Five played Radiant; they won
    },
    'coop': {
        'match_id':     4132076146,
        'label':        'OpenAI Five Coop Game',
        'radiant_win':  True,
        'radiant':      {'gyrocopter','lion','sven','sniper','viper'},
        'dire':         {'crystal_maiden','death_prophet','earthshaker','queenofpain','razor'},
        'ai_side':      'both',    # Both teams are OpenAI Five
    },
}

def detect_buybacks(losses_raw, all_heroes, game_start):
    """Two looses events within 0.5s for same hero = one buyback.
    The larger amount is buyback cost; smaller is death penalty."""
    hero_losses = defaultdict(list)
    for t, hero, amt in losses_raw:
        if hero in all_heroes:
            hero_losses[hero].append((t, amt))
    buybacks = []
    for hero, events in hero_losses.items():
        events.sort()
        i = 0
        while i < len(events):
            t0, amt0 = events[i]
            if i+1 < len(events) and abs(events[i+1][0] - t0) < 0.6:
                buybacks.append({
                    'game_time':    round(t0 - game_start, 1),
                    'hero':         short(hero),
                    'buyback_cost': max(amt0, events[i+1][1]),
                    'death_penalty':min(amt0, events[i+1][1]),
                })
                i += 2
            else:
                i += 1
    return sorted(buybacks, key=lambda x: x['game_time'])

def parse_combatlog(path, cfg):
    """Parse a single combatlog file and return structured game data."""
    radiant_s = {'npc_dota_hero_' + h for h in cfg['radiant']}
    dire_s    = {'npc_dota_hero_' + h for h in cfg['dire']}
    all_h     = radiant_s | dire_s

    game_start = game_end = None
    kills = []
    gold_events = []   # (rec_time, hero, delta)
    losses_raw  = []   # (rec_time, hero, amount) for buyback detection

    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            m = STATE_RE.search(line)
            if m:
                t = ts2s(*m.groups()[:4])
                if m.group(5) == '4' and game_start is None: game_start = t
                if m.group(5) == '6' and game_end   is None: game_end   = t
                continue
            m = KILL_RE.search(line)
            if m:
                t, victim, killer = ts2s(*m.groups()[:4]), m.group(5), m.group(6)
                if victim in all_h and killer in all_h:
                    kills.append({
                        'rec_time':      t,
                        'victim':        victim,
                        'attacker':      killer,
                        'victim_team':   'radiant' if victim in radiant_s else 'dire',
                        'attacker_team': 'radiant' if killer in radiant_s else 'dire',
                    })
                continue
            m = RECV_RE.search(line)
            if m:
                t, hero, amt = ts2s(*m.groups()[:4]), m.group(5), int(m.group(6))
                if hero in all_h: gold_events.append((t, hero, +amt))
                continue
            m = LOSS_RE.search(line)
            if m:
                t, hero, amt = ts2s(*m.groups()[:4]), m.group(5), int(m.group(6))
                if hero in all_h:
                    losses_raw.append((t, hero, amt))
                    gold_events.append((t, hero, -amt))

    if game_start is None:
        raise ValueError(f'No game start found in {path}')

    duration = round(game_end - game_start, 1) if game_end else None
    for k in kills:
        k['game_time'] = round(k['rec_time'] - game_start, 1)

    # --- Gold timeline (60s intervals) ---
    gold = {h: 600 for h in all_h}
    gold_events_sorted = sorted(gold_events, key=lambda x: x[0])
    timeline = []
    ei = 0
    for i in range(int(duration // 60) + 3 if duration else 45):
        sample_rec = game_start + i * 60
        if duration and i * 60 > duration + 60: break
        while ei < len(gold_events_sorted) and gold_events_sorted[ei][0] <= sample_rec:
            _, hero, delta = gold_events_sorted[ei]
            gold[hero] = max(0, gold[hero] + delta)
            ei += 1
        rad  = sum(gold[h] for h in radiant_s)
        dire = sum(gold[h] for h in dire_s)
        timeline.append({'game_time': i*60, 'radiant_gold': rad,
                         'dire_gold': dire, 'gold_adv': rad - dire})

    # --- Teamfight clustering (20s gap) ---
    sorted_kills = sorted(kills, key=lambda x: x['rec_time'])
    clusters, cur = [], []
    if sorted_kills:
        cur = [sorted_kills[0]]
        for k in sorted_kills[1:]:
            if k['rec_time'] - cur[-1]['rec_time'] <= 20: cur.append(k)
            else: clusters.append(cur); cur = [k]
        clusters.append(cur)

    teamfights = [{
        'start_time':     c[0]['game_time'],
        'end_time':       c[-1]['game_time'],
        'radiant_deaths': sum(1 for k in c if k['victim_team'] == 'radiant'),
        'dire_deaths':    sum(1 for k in c if k['victim_team'] == 'dire'),
    } for c in clusters]

    buybacks = detect_buybacks(losses_raw, all_h, game_start)

    print(f"  {cfg['label']}: {len(kills)} kills, {len(teamfights)} fights, "
          f"{len(buybacks)} buybacks, {duration:.0f}s")

    return {
        'cfg':        cfg,
        'duration':   duration,
        'timeline':   timeline,
        'teamfights': teamfights,
        'buybacks':   buybacks,
    }

def get_gold_adv_at(timeline, game_time, team):
    """Look up gold advantage at a given game time (60s resolution)."""
    idx = min(int(game_time / 60), len(timeline) - 1)
    adv = timeline[idx]['gold_adv']
    return adv if team == 'radiant' else -adv

print('Parser functions defined.')

In [ ]:
# ============================================================
# CELL 3 — Parse all 3 games and generate AI shock events
# ============================================================
ai_rows = []

print('Parsing combatlog files...')
for game_id, cfg in GAME_CFG.items():
    path = COMBATLOG_PATHS.get(game_id)
    if path is None:
        print(f'  SKIP {game_id}: no file uploaded')
        continue

    game = parse_combatlog(path, cfg)
    timeline    = game['timeline']
    teamfights  = game['teamfights']
    radiant_win = cfg['radiant_win']
    ai_side     = cfg['ai_side']
    match_id    = cfg['match_id']

    ai_teams = ['radiant', 'dire'] if ai_side == 'both' else [ai_side]

    for tf in teamfights:
        r_deaths = tf['radiant_deaths']
        d_deaths = tf['dire_deaths']
        if r_deaths == 0 and d_deaths == 0: continue

        for team in ['radiant', 'dire']:
            my_net = (d_deaths - r_deaths) if team == 'radiant' else (r_deaths - d_deaths)
            if my_net < 0:   shock_type = 'negative'
            elif my_net > 0: shock_type = 'positive'
            else:            continue

            is_ai  = 1 if team in ai_teams else 0
            did_win = 1 if ((team == 'radiant') == radiant_win) else 0
            fight_start = tf['start_time']
            base_g = get_gold_adv_at(timeline, fight_start, team)

            yields = {}
            for h in HORIZONS:
                future_g = get_gold_adv_at(timeline, fight_start + h, team)
                yields[f'gadv_d{h}'] = int(future_g - base_g)

            ai_rows.append({
                'match_id':          match_id,
                'shock_type':        shock_type,
                'is_ai':             is_ai,
                'did_win':           did_win,
                'fight_start':       int(fight_start),
                'current_gold_lead': int(base_g),
                **yields,
            })

ai_df = pd.DataFrame(ai_rows)
print(f'\nAI rows generated: {len(ai_df)}')
print(f'  is_ai=1: {(ai_df["is_ai"]==1).sum()}')
print(f'  is_ai=0: {(ai_df["is_ai"]==0).sum()}  (non-AI team in each match)')
print(ai_df[ai_df['is_ai']==1].groupby(['match_id','shock_type']).size().to_string())

In [ ]:
# ============================================================
# CELL 4 — Download human baseline from OpenDota API (Part B)
# ============================================================
# TI8 league_id = 9870, TI9 league_id = 10749
# We fetch Main Event matches only and extract tactical shock events.

TI_LEAGUES   = {9870: 'TI8', 10749: 'TI9'}
OPENDOTA_API = 'https://api.opendota.com/api'
OPENAI_IDS   = {4693543125, 4693543123, 4693543117, 4080601137, 4080420268, 4080316101}

def get_gold_at_time(gold_adv_list, time_sec):
    """Same 60s-resolution lookup used in original prep script."""
    if not gold_adv_list: return 0
    idx = int(time_sec / 60)
    return gold_adv_list[idx] if idx < len(gold_adv_list) else gold_adv_list[-1]

def fetch_league_matches(league_id):
    """Fetch all match IDs for a league from OpenDota."""
    url = f'{OPENDOTA_API}/leagues/{league_id}/matches'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return [m['match_id'] for m in resp.json()]

def fetch_match(match_id):
    """Fetch full match data from OpenDota."""
    url = f'{OPENDOTA_API}/matches/{match_id}'
    resp = requests.get(url, timeout=30)
    if resp.status_code == 200:
        return resp.json()
    return None

def parse_opendota_match(data):
    """Extract tactical shock events from an OpenDota match JSON."""
    match_id    = data.get('match_id')
    radiant_win = data.get('radiant_win')
    gold_adv    = data.get('radiant_gold_adv', [])
    if not gold_adv: return []

    events = []
    for tf in data.get('teamfights', []):
        start    = tf['start']
        players  = tf.get('players', [])
        r_deaths = sum(p.get('deaths', 0) for i,p in enumerate(players) if i < 5)
        d_deaths = sum(p.get('deaths', 0) for i,p in enumerate(players) if i >= 5)
        if r_deaths == 0 and d_deaths == 0: continue

        for team in ['radiant', 'dire']:
            my_net = (d_deaths - r_deaths) if team == 'radiant' else (r_deaths - d_deaths)
            if my_net < 0:   shock_type = 'negative'
            elif my_net > 0: shock_type = 'positive'
            else:            continue

            did_win   = int((radiant_win and team=='radiant') or (not radiant_win and team=='dire'))
            base_g    = get_gold_at_time(gold_adv, start)
            cur_lead  = base_g if team == 'radiant' else -base_g

            yields = {}
            for h in HORIZONS:
                change = get_gold_at_time(gold_adv, start + h) - base_g
                yields[f'gadv_d{h}'] = change if team == 'radiant' else -change

            events.append({
                'match_id':          match_id,
                'shock_type':        shock_type,
                'is_ai':             0,
                'did_win':           did_win,
                'fight_start':       start,
                'current_gold_lead': cur_lead,
                **yields,
            })
    return events

print('OpenDota functions defined.')
print('Cell 5 will fetch the human baseline data (~10-15 min depending on API rate limits).')

In [ ]:
# ============================================================
# CELL 5 — Fetch TI8 + TI9 matches from OpenDota
# NOTE: This cell makes ~500+ API calls. OpenDota rate-limits
# to 60 req/min for unauthenticated users. Expect ~15 minutes.
# If you already have tf_events_master_unified_v2.csv, skip to Cell 7.
# ============================================================
human_rows  = []
failed_ids  = []
all_match_ids = []

print('Fetching match lists for TI8 and TI9...')
for league_id, label in TI_LEAGUES.items():
    try:
        ids = fetch_league_matches(league_id)
        # Filter out OpenAI matches (already handled separately)
        ids = [i for i in ids if i not in OPENAI_IDS]
        print(f'  {label}: {len(ids)} matches')
        all_match_ids.extend(ids)
    except Exception as e:
        print(f'  {label}: FAILED — {e}')

print(f'\nTotal matches to fetch: {len(all_match_ids)}')
print('Fetching match data...')

for i, mid in enumerate(all_match_ids):
    try:
        data = fetch_match(mid)
        if data:
            rows = parse_opendota_match(data)
            human_rows.extend(rows)
        # Respect OpenDota rate limit: 1 req/sec to be safe
        time.sleep(1.0)
        if (i+1) % 50 == 0:
            print(f'  {i+1}/{len(all_match_ids)} matches — {len(human_rows)} events so far')
    except Exception as e:
        failed_ids.append(mid)

human_df = pd.DataFrame(human_rows)
print(f'\nHuman rows: {len(human_df)}')
if failed_ids:
    print(f'Failed to fetch: {len(failed_ids)} matches')

In [ ]:
# ============================================================
# CELL 6 — ALTERNATIVE: Upload existing human baseline CSV
# If you already have tf_events_master_unified_v2.csv or the
# original tf_events_master_unified.csv, upload it here instead
# of running Cell 5.
# ============================================================
USE_EXISTING_CSV = True  # Set to False if you ran Cell 5

if USE_EXISTING_CSV:
    print('Upload your existing human baseline CSV:')
    print('(tf_events_master_unified.csv or tf_events_master_unified_v2.csv)')
    up = files.upload()
    existing_path = list(up.keys())[0]
    with open(f'/content/{existing_path}', 'wb') as f:
        f.write(up[existing_path])

    existing_df = pd.read_csv(f'/content/{existing_path}')
    # Keep only human rows — we'll replace AI rows with our parsed data
    human_df = existing_df[existing_df['is_ai'] == 0].copy()
    print(f'Loaded {len(human_df)} human rows from existing CSV.')
else:
    print(f'Using {len(human_df)} human rows fetched from OpenDota API.')

In [ ]:
# ============================================================
# CELL 7 — Merge AI + human rows into unified dataset
# ============================================================
combined = pd.concat([human_df, ai_df], ignore_index=True)

# Ensure correct dtypes
combined['is_ai']    = combined['is_ai'].astype(int)
combined['did_win']  = combined['did_win'].astype(int)
combined['match_id'] = combined['match_id'].astype(int)

print('Dataset summary:')
print(f'  Total rows:   {len(combined):,}')
print(f'  AI rows:      {(combined["is_ai"]==1).sum()}')
print(f'  Human rows:   {(combined["is_ai"]==0).sum()}')
print(f'  Shock types:  {combined["shock_type"].value_counts().to_dict()}')
print(f'  Columns:      {list(combined.columns)}')
print()
print('AI rows by match:')
print(combined[combined['is_ai']==1].groupby('match_id')['shock_type']
      .value_counts().to_string())

# Save
out_path = f'{OUT_DIR}/tf_events_master_unified_v2.csv'
combined.to_csv(out_path, index=False)
print(f'\n✅ Saved: {out_path}')

In [ ]:
# ============================================================
# CELL 8 — Download the final dataset
# ============================================================
files.download(f'{OUT_DIR}/tf_events_master_unified_v2.csv')
print('Downloaded tf_events_master_unified_v2.csv')
print('This file is the input for Step 3 (analysis notebook).')